## Get Meeting Room Availability

Need to get an updated cookie before using

In [1]:
from pathlib import Path
import json
import os
import sys
sys.path.insert(0, 'src')

# Load secrets from .env so they stay out of the notebook itself
try:
    from dotenv import load_dotenv

    load_dotenv(Path('.env'))
except Exception as exc:
    print(f"Could not load .env automatically: {exc}")


def parse_resource_ids(raw_value):
    """Accept comma-separated or JSON list strings for RESOURCE_IDS."""
    if not raw_value:
        return []
    try:
        parsed = json.loads(raw_value)
        if isinstance(parsed, list):
            return [str(item) for item in parsed]
    except Exception:
        pass
    return [part.strip() for part in raw_value.split(',') if part.strip()]


cookie = os.getenv("DISH_COOKIE")
team_id = os.getenv("TEAM_ID")
member_id = os.getenv("MEMBER_ID")
resource_ids = parse_resource_ids(os.getenv("RESOURCE_IDS"))

if not resource_ids:
    resource_ids = [
        "6422bced61d5854ab3fedd62",  # Boyle
        "6422bcd50340a914e68e661b",  # Pankhurst
        "6422bcff9814c9c32ed62d77",  # Turing
    ]

if not cookie:
    raise ValueError("Set DISH_COOKIE in your .env before running this notebook.")

shared_params = {
    "resource_ids": resource_ids,
    "cookie": cookie,
    "team_id": team_id,
    "member_id": member_id,
}


In [ ]:
from src.get_room_availability import get_room_availability
import json
from src.utils.type_defs import DatetimeRange

availability_params = {
    "resource_ids": shared_params["resource_ids"],
    "start": "2025-11-27T16:00:00.000Z",
    "end": "2025-11-27T17:00:00.000Z",
    "cookie": shared_params["cookie"],
}

response = get_room_availability(
    resource_ids=availability_params["resource_ids"],
    datetime_range=DatetimeRange(
        start_datetime=availability_params["start"],
        end_datetime=availability_params["end"],
    ),
    cookie=availability_params["cookie"],
)

print(f"Status Code: {response.status_code}")
print("Response:")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))


Status Code: 200
Response:
[
  {
    "_id": "691c892654ee8d45333eb4a8",
    "targetType": "booking",
    "target": "691c892654ee8d45333eb49d",
    "start": {
      "dateTime": "2025-11-27T14:30:00.000Z"
    },
    "end": {
      "dateTime": "2025-11-27T16:30:00.000Z"
    },
    "isShadow": false,
    "serviceSlots": {
      "before": 0,
      "after": 0
    },
    "recurrence": {
      "rrule": null
    },
    "resourceId": {
      "_id": "6422bced61d5854ab3fedd62",
      "parents": [],
      "type": "meeting_room",
      "office": "641093286b69a186501888d3",
      "name": "Boyle"
    },
    "team": null,
    "member": null,
    "createdAt": "2025-11-18T14:56:38.295Z",
    "createdBy": "6448ec22f88e8b70653c990c",
    "timezone": "Europe/London",
    "source": "portal",
    "summary": "Busy",
    "accountedUntil": null,
    "tentative": false,
    "accounted": true,
    "reference": "21N5374",
    "office": "641093286b69a186501888d3",
    "bookingId": "691c892654ee8d45333eb49d"
  },
  {

### Extract Room Availability from the response

**Important API Behaviour:**

The API only returns bookings that exist during the time range specified in the query parameters. This means:

- **If a room has bookings**: The room will appear in the response with its booking details
- **If a room has no bookings**: The room will **not** appear in the response at all
- **If all rooms are free**: The API returns an empty array `[]`

Therefore, to determine complete availability, we need to:
1. Know which rooms we're querying (from the `resourceId` parameter)
2. Check which rooms appear in the bookings response
3. Any rooms that don't appear are completely free for the entire time range

This function handles this by:
- Accepting an optional list of queried room IDs/names
- When bookings is empty, marking all queried rooms as completely available
- When bookings exist, processing gaps between bookings as available slots

In [3]:
from src.get_room_availability import display_room_availability

availability = display_room_availability(
    response=response,
    datetime_range=DatetimeRange(
        start_datetime=availability_params["start"],
        end_datetime=availability_params["end"],
    ),
    queried_room_ids=availability_params["resource_ids"]
)


ROOM AVAILABILITY SUMMARY

Boyle:
  Total Available: 30 minutes
  Total Booked: 120 minutes
  Available Slots: 1
  Booked Slots: 1

  Available Time Slots:
    2025-11-27 16:30 - 17:00 (30 min)

  Booked Time Slots:
    2025-11-27 14:30 - 16:30 - Busy (booking ID: 691c892654ee8d45333eb49d)

Pankhurst:
  Total Available: 0 minutes
  Total Booked: 60 minutes
  Available Slots: 0
  Booked Slots: 1

  Booked Time Slots:
    2025-11-27 16:00 - 17:00 - Busy (booking ID: 6904df0e87263fa36f9dba8c)

Turing:
  Note: No bookings found - room completely available
  Total Available: 60 minutes
  Total Booked: 0 minutes
  Available Slots: 1
  Booked Slots: 0

  Available Time Slots:
    2025-11-27 16:00 - 17:00 (60 min)


## Book a Meeting Room

WARNING: This will book a room for the current user.

In [2]:
booking_params = {
    "resource_ids": shared_params["resource_ids"],
    "start": "2025-11-26T21:00:00.000Z",
    "end": "2025-11-26T22:00:00.000Z",
    "cookie": shared_params["cookie"],
    "team_id": shared_params["team_id"],
    "member_id": shared_params["member_id"],
    "summary": "Booking",
    "meeting_room_name": os.getenv("MEETING_ROOM_NAME", "Boyle"),
}

missing = [key for key in ("team_id", "member_id") if not booking_params[key]]
if missing:
    raise ValueError(f"Set {', '.join(missing)} in your .env before booking.")


In [4]:
from src.book_room import book_room, format_booking_response
from src.utils.type_defs import UserInfo, DatetimeRange
import json

response = book_room(
    datetime_range=DatetimeRange(
        start_datetime=booking_params["start"],
        end_datetime=booking_params["end"],
    ),
    meeting_room_name=booking_params["meeting_room_name"],
    user_info=UserInfo(
        team_id=booking_params["team_id"],
        member_id=booking_params["member_id"],
    ),
    cookie=booking_params["cookie"],
    summary=booking_params["summary"],
)

formatted_response = format_booking_response(response.json())
print(json.dumps(formatted_response, indent=2))


{
  "createdAt": "2025-11-26T18:36:38.451Z",
  "createdBy": "6745f545c44869c9ee21a29c",
  "organization": "641092905ecc7d53af9cbd2c",
  "start": {
    "dateTime": "2025-11-26T21:00:00.000Z"
  },
  "end": {
    "dateTime": "2025-11-26T22:00:00.000Z"
  },
  "serviceSlots": {
    "before": 0,
    "after": 0
  },
  "timezone": "Europe/London",
  "source": "portal",
  "summary": "Booking",
  "seriesStart": "2025-11-26T21:00:00.000Z",
  "seriesEnd": "2025-11-26T22:00:00.000Z",
  "recurrence": {
    "rrule": null
  },
  "fees": [],
  "resourceId": "6422bced61d5854ab3fedd62",
  "team": "641c24c7dcaed13e1d766ee2",
  "member": "6745f5302bc6e87caa0c15ae",
  "visitors": [],
  "members": [],
  "accountedUntil": null,
  "tentative": false,
  "accounted": true,
  "reference": "F2W5274",
  "office": "641093286b69a186501888d3",
  "_id": "692748b6d4690493e2b6c818",
  "modifiedAt": "2025-11-26T18:36:38.451Z",
  "modifiedBy": "6745f545c44869c9ee21a29c",
  "title": "Booked Boyle from 2025-11-26 21:00:00 to

## Get booking 